# 在 Notebook 中提问 LightRAG

这个 notebook 用来直接调用项目的 `/chat/stream` 流式接口，不用手写 `curl`。

启动方式：

```bash
python -m agentic_rag.main serve
```

默认 API 地址按 `APP_PORT=8010` 处理，避免和 `.env` 里的 vLLM `8000` 冲突。

In [ ]:
import html
import json
import re
from pathlib import Path
from pprint import pprint

import pandas as pd
import requests
from IPython.display import HTML, display

API_BASE = "http://127.0.0.1:8010"
TRIPLES_PATH = Path("../lightrag/standard_triples.json")


def _iter_sse_events(response):
    for raw_line in response.iter_lines(chunk_size=1, decode_unicode=True):
        if not raw_line:
            continue
        if not raw_line.startswith("data: "):
            continue
        payload = raw_line[6:]
        if not payload.strip():
            continue
        yield json.loads(payload)


def _strip_think_blocks(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()


def _render_stream_panel(route, reason, status_lines, answer_text):
    route_html = html.escape(route or "-")
    reason_html = html.escape(reason or "-")
    status_html = "<br>".join(html.escape(line) for line in status_lines) or "等待返回..."
    answer_html = html.escape(answer_text or "").replace("\n", "<br>")
    return HTML(
        f'''
        <div style="border:1px solid #d0d7de;border-radius:8px;padding:12px 14px;margin:8px 0;">
          <div style="margin-bottom:8px;"><b>路由:</b> {route_html}</div>
          <div style="margin-bottom:8px;"><b>原因:</b> {reason_html}</div>
          <div style="margin-bottom:8px;"><b>状态:</b><br>{status_html}</div>
          <div><b>回答:</b></div>
          <div style="white-space:pre-wrap;line-height:1.6;font-family:Consolas, 'Courier New', monospace;max-height:420px;overflow:auto;">{answer_html}</div>
        </div>
        '''
    )


def ask(question: str, force_route: str = "lightrag", hide_thinking: bool = True):
    payload = {
        "question": question,
        "force_route": force_route,
    }
    answer_parts = []
    route = None
    reason = None
    status_lines = ["正在连接流式接口..."]
    panel = display(_render_stream_panel(route, reason, status_lines, ""), display_id=True)

    def refresh():
        current_answer = "".join(answer_parts)
        if hide_thinking:
            current_answer = _strip_think_blocks(current_answer)
        panel.update(_render_stream_panel(route, reason, status_lines, current_answer))

    with requests.post(
        f"{API_BASE}/chat/stream",
        json=payload,
        stream=True,
        timeout=(10, None),
        headers={"Accept": "text/event-stream"},
    ) as response:
        response.raise_for_status()
        for event in _iter_sse_events(response):
            event_type = event.get("type")
            if event_type == "ready":
                status_lines = ["已连接到流式接口"]
                refresh()
            elif event_type == "meta":
                route = event.get("route")
                reason = event.get("reason")
                refresh()
            elif event_type == "status":
                message = event.get("message", "")
                if message:
                    status_lines.append(message)
                    status_lines = status_lines[-3:]
                refresh()
            elif event_type == "delta":
                chunk = event.get("content", "")
                if chunk:
                    answer_parts.append(chunk)
                    refresh()
            elif event_type == "error":
                status_lines.append(f"错误: {event.get('message') or '流式接口返回错误。'}")
                refresh()
                raise RuntimeError(event.get("message") or "流式接口返回错误。")
            elif event_type == "done":
                route = route or event.get("route")
                reason = reason or event.get("reason")
                status_lines.append("流式输出完成")
                refresh()

    answer = "".join(answer_parts)
    if hide_thinking:
        answer = _strip_think_blocks(answer)
    return {
        "answer": answer.strip(),
        "route": route,
        "reason": reason,
    }


def load_triples(path: Path = TRIPLES_PATH):
    if not path.exists():
        raise FileNotFoundError(f"未找到标准三元组文件: {path.resolve()}")
    return json.loads(path.read_text(encoding="utf-8"))


def triples_df(path: Path = TRIPLES_PATH):
    triples = load_triples(path)
    return pd.DataFrame(triples)


def search_triples(keyword: str, path: Path = TRIPLES_PATH, limit: int = 20):
    df = triples_df(path)
    lowered = keyword.lower()
    mask = (
        df["subject"].str.lower().str.contains(lowered, na=False)
        | df["predicate"].str.lower().str.contains(lowered, na=False)
        | df["object"].str.lower().str.contains(lowered, na=False)
        | df["evidence"].str.lower().str.contains(lowered, na=False)
    )
    return df.loc[mask].head(limit)


In [ ]:
# 先测一下服务是否正常
requests.get(f"{API_BASE}/health", timeout=30).json()

In [ ]:
ask("Who publishes ASME B16.5-2013?")

In [ ]:
ask("What is Table II-9 about in ASME B16.5-2013?")

In [ ]:
ask("What does ASME B16.5-2013 say about ring joint facings?")

## 查看标准三元组

重新执行入库后，项目会在 `lightrag/standard_triples.json` 中写入标准谓语三元组。下面几个单元格用来检查导出结果。

In [ ]:
# 查看文件是否存在，以及总共有多少条三元组
triples = load_triples()
print("标准三元组文件:", TRIPLES_PATH.resolve())
print("三元组总数:", len(triples))

In [ ]:
# 随机看前几条
pprint(triples[:5])

In [ ]:
# 以表格形式查看，便于筛选
df = triples_df()
df.head(10)

In [ ]:
# 按关键词筛选，例如查询 ASME B16.5-2013 的相关三元组
search_triples("ASME B16.5-2013")

In [ ]:
# 查询温度和材料相关的标准三元组
search_triples("Graphite")